In [2]:
import pandas as pd
import glob
import passiogo

USU_CAMPUS_BUS = 3499
# system = passiogo.getSystemFromID(USU_CAMPUS_BUS)

# routes = system.getRoutes()
# display(list(map(lambda route: f"{route.id}: {route.name}", routes)))

# buses = system.getVehicles()
# display(list(map(lambda bus: f"{bus.routeId}: {bus.routeName}", buses)))

# stops = system.getStops()
# named_stops = list(map(lambda stop: (stop.name, (stop.longitude, stop.latitude), list(stop.routesAndPositions.keys())), stops))
# # print(named_stops)
# display(named_stops)

In [ ]:
semester = "fall26"
date = "10-1-26"
all_files = glob.glob(f"out/bus/{semester}/bus_{date}_*")

col_names = ["timestamp"] + [f"extra_col_{i}" for i in range(1, 10)]
df_list = (pd.read_csv(f, names=col_names, dtype=str, header=None) for f in all_files)
df = pd.concat(df_list, ignore_index=True)

cols = df.items()
next(cols)

labeled_vehicle_positions = pd.DataFrame(columns=['timestamp', 'vehicle_id', 'route_id', 'coordinates'])

def lat_long_to_tuple(a):
    latitude = a['latitude']
    longitude = a['longitude']

    if latitude == None or longitude == None:
        return None
    else:
        return (float(longitude), float(latitude))

for col_name, col_data in cols:
    vehicle_data = col_data.str.split(pat=r"[\/\@;]", expand=True)
    if len(vehicle_data.columns) != 4:
        continue

    vehicle_data.columns = ['vehicle_id', 'route_id', 'latitude', 'longitude']

    timestamps = df.iloc[:, 0].astype(float)
    timestamps.name = 'timestamp'

    vehicle_data = pd.concat([timestamps, vehicle_data], axis=1)
    vehicle_data.dropna()

    vehicle_data['coordinates'] = vehicle_data[['latitude', 'longitude']].apply(lat_long_to_tuple, axis=1)
    vehicle_data.drop(['latitude', 'longitude'], axis=1, inplace=True)

    labeled_vehicle_positions = pd.concat([labeled_vehicle_positions, vehicle_data])

labeled_vehicle_positions = labeled_vehicle_positions[labeled_vehicle_positions.route_id.isin({'38765', '38768', '53861'})]
labeled_vehicle_positions

,timestamp,vehicle_id,route_id,coordinates
129,1790903020.051518,14327,38765,"(-111.8126951, 41.7552911)"
130,1790903030.304711,14327,38765,"(-111.8126878, 41.7552878)"
131,1790903040.56234,14327,38765,"(-111.8126721, 41.7552863)"
132,1790903050.821153,14327,38765,"(-111.8126641, 41.7552763)"
133,1790903061.086387,14327,38765,"(-111.8126585, 41.7552676)"
...,...,...,...,...
2596,1790891377.109421,18512,38765,"(-111.8141216, 41.7461836)"
2597,1790891387.364824,18512,38765,"(-111.8141521, 41.7450158)"
2598,1790891397.639757,18512,38765,"(-111.8141476, 41.7447991)"
2599,1790891407.897185,18512,38765,"(-111.814152, 41.7446456)"


In [7]:
from route import Route
from shapely import Point
import shapely.ops
import pyproj
import usu_routes

transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32142", always_xy=True).transform
usu_stops_transformed = list(map(lambda stop: (stop[0], shapely.ops.transform(transformer, Point(stop[1])), stop[2]), usu_routes.SYSTEM_STOPS))

AT_STOP_THRESHOLD_M = 16

green_line_segments = usu_routes.import_route(usu_routes.GREEN_ROUTE)
green_line = Route(green_line_segments)

routes = {
    '38765': green_line,
    '38768': None,
    '53861': None
}

def dist_marker(a):
    route = routes.get(a['route_id'], None)
    if route == None: return None
    point = Point(a['coordinates'])
    return next(route.distance_marker_for(point))

def stop_name(a):
    point = Point(a['coordinates'])
    point_transformed = shapely.ops.transform(transformer, point)
    route = a['route_id']
    return next((stop[0] for stop in usu_stops_transformed if route in stop[2] and point_transformed.distance(stop[1]) <= AT_STOP_THRESHOLD_M), "in_transit")

labeled_vehicle_positions["stop"] = labeled_vehicle_positions[['coordinates', 'route_id']].apply(stop_name, axis=1)
# labeled_vehicle_positions["distance_along_route"] = labeled_vehicle_positions[['coordinates', 'route_id']].apply(dist_marker, axis=1)

south_campus = labeled_vehicle_positions[labeled_vehicle_positions.route_id.isin({'38768', '53861'})].copy()
south_campus.drop(columns='route_id', inplace=True)
south_campus.to_csv(f"data/{semester}/south_campus_{date}.csv", index=False)

housing_express = labeled_vehicle_positions[labeled_vehicle_positions.route_id == '38765'].copy()
housing_express.drop(columns='route_id', inplace=True)
housing_express.to_csv(f"data/{semester}/housing_express_{date}.csv", index=False)